# ACLED Africa: spatiotemporal grid model for protest diffusion

**Research question:** How do protests spread over time and space in Africa?

This notebook transforms ACLED protest events in Africa, January 2000 to December 2020, into a **grid-cell x month** panel. The main specification uses a **100 x 100 km grid**, with robustness checks at **50, 100, and 200 km**.

Hypotheses:
- **H1 Spatial diffusion:** protests in nearby grid cells raise risk in the focal cell, even after accounting for the focal cell's own 1-month, 2-month, 6-month, and 1-year protest history.
- **H2 Scale robustness:** the same broad pattern should be visible when the grid is made smaller (50 x 50 km) or larger (200 x 200 km).
- **H3 Fatality heterogeneity:** nonfatal protests produce stronger spillover effects than fatal protests.

Temporal split:
- 2000-2015: training period
- 2016-2020: held-out test period

Project layout:
- `data/new data/`: protest-only training and test CSV files
- `notebooks/`: this research notebook
- `outputs/tables/`: model summaries and evaluation tables
- `outputs/figures/`: saved charts also displayed inline by the notebook
- `outputs/panels/`: generated cell-month panels and test predictions


## 0. Installation

Run the cell below only if your VSCode kernel is missing the required packages.

In [ ]:
# Run if needed in VSCode:
# %pip install pandas numpy matplotlib scikit-learn seaborn nbformat

In [ ]:
from pathlib import Path
import math
import os
import sys
import subprocess
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

try:
    from IPython.display import display, Image
except Exception:
    Image = None
    def display(x):
        print(x)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, precision_recall_fscore_support, precision_recall_curve
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.calibration import calibration_curve

REPO_URL = "https://github.com/Mat99999/acled-spatiotemporal-conflict-model.git"  # Used by Colab to clone the full project when needed.
PROJECT_FOLDER_NAME = "acled_spatiotemporal_project"
DATA_SUBDIR = Path("data") / "new data"
TRAIN_DATA_FILENAME = "Protest_2000_2015_train.csv"
TEST_DATA_FILENAME = "Protest_2016_2020_test.csv"


def running_in_colab():
    return "google.colab" in sys.modules


def find_project_dir():
    """Find the project root without hard-coded local paths."""
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent, cwd / PROJECT_FOLDER_NAME, cwd.parent / PROJECT_FOLDER_NAME]
    for candidate in candidates:
        if (candidate / DATA_SUBDIR / TRAIN_DATA_FILENAME).exists() and (candidate / DATA_SUBDIR / TEST_DATA_FILENAME).exists():
            return candidate

    # If opened directly in Colab from GitHub, the files are not automatically cloned.
    if running_in_colab() and REPO_URL:
        clone_dir = Path("/content") / PROJECT_FOLDER_NAME
        if not clone_dir.exists():
            subprocess.run(["git", "clone", REPO_URL, str(clone_dir)], check=True)
        return clone_dir

    raise FileNotFoundError(
        "Could not locate the project folder. Run this notebook from the cloned "
        f"{PROJECT_FOLDER_NAME} repository root, or keep the notebook inside the "
        "project's notebooks/ folder. Expected data files under "
        f"{DATA_SUBDIR}: {TRAIN_DATA_FILENAME} and {TEST_DATA_FILENAME}."
    )


PROJECT_DIR = find_project_dir()
DATA_DIR = PROJECT_DIR / DATA_SUBDIR
TRAIN_DATA_PATH = DATA_DIR / TRAIN_DATA_FILENAME
TEST_DATA_PATH = DATA_DIR / TEST_DATA_FILENAME
OUTPUT_DIR = PROJECT_DIR / "outputs"
TABLES_DIR = OUTPUT_DIR / "tables"
FIGURES_DIR = OUTPUT_DIR / "figures"
PANELS_DIR = OUTPUT_DIR / "panels"
for directory in [TABLES_DIR, FIGURES_DIR, PANELS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


def save_and_show(fig, filename, dpi=140):
    path = FIGURES_DIR / filename
    fig.savefig(path, dpi=dpi)
    if Image is not None:
        display(Image(filename=str(path)))
    else:
        print(f"Saved figure: {path}")
    plt.close(fig)


RANDOM_STATE = 42
PRIMARY_GRID_KM = 100
ROBUSTNESS_GRID_KM = [50, 100, 200]
ANALYSIS_START = pd.Timestamp('2000-01-01')
ANALYSIS_END = pd.Timestamp('2020-12-31')
TRAIN_END = pd.Timestamp('2015-12-31')
TEST_START = pd.Timestamp('2016-01-01')
TEST_END = pd.Timestamp('2020-12-31')

print('Training data:', TRAIN_DATA_PATH)
print('Test data:', TEST_DATA_PATH)
print('Project:', PROJECT_DIR)
print('Output:', OUTPUT_DIR)


## 1. Load and validate ACLED protest data

The source data is protest-only ACLED event data. The raw 2000-2020 file has been split before modeling so the notebook uses a forward-looking training/test design.


In [ ]:
train_events = pd.read_csv(TRAIN_DATA_PATH)
test_events = pd.read_csv(TEST_DATA_PATH)
train_events['split'] = 'train'
test_events['split'] = 'test'

df = pd.concat([train_events, test_events], ignore_index=True)
df.columns = [c.strip().lower() for c in df.columns]

df['event_date'] = pd.to_datetime(df['event_date'], errors='coerce')
df['year'] = pd.to_numeric(df['year'], errors='coerce').astype('Int64')
df['fatalities'] = pd.to_numeric(df['fatalities'], errors='coerce').fillna(0)
df['fatal_protest'] = pd.to_numeric(df.get('fatal_protest', (df['fatalities'] >= 1).astype(int)), errors='coerce').fillna(0).astype(int)
df['latitude'] = pd.to_numeric(df['latitude'], errors='coerce')
df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')
df['month'] = df['event_date'].dt.to_period('M').dt.to_timestamp()

required = ['event_date','month','latitude','longitude','country','event_type','sub_event_type','fatalities','fatal_protest','split']
print('Rows:', len(df))
print('Date range:', df['event_date'].min(), 'to', df['event_date'].max())
print('Training rows:', (df['split'] == 'train').sum(), '| Test rows:', (df['split'] == 'test').sum())
print('Event types:', df['event_type'].dropna().unique())
display(df[required].isna().sum().to_frame('missing'))
display(df.head(3))


In [ ]:
summary = pd.DataFrame({
    'metric': ['protest_events','countries','months','fatalities_total','fatal_protest_events','share_nonfatal_protests'],
    'value': [
        len(df),
        df['country'].nunique(),
        df['month'].nunique(),
        int(df['fatalities'].sum()),
        int(df['fatal_protest'].sum()),
        round((df['fatal_protest'] == 0).mean(), 3),
    ]
})
by_type = df['event_type'].value_counts().rename_axis('event_type').reset_index(name='events')
by_subtype = df['sub_event_type'].value_counts().rename_axis('sub_event_type').reset_index(name='events')
by_split = df.groupby('split').agg(
    events=('event_id_cnty', 'count'),
    months=('month', 'nunique'),
    countries=('country', 'nunique'),
    fatal_protest_events=('fatal_protest', 'sum'),
).reset_index()

display(summary)
display(by_type)
display(by_subtype)
display(by_split)
summary.to_csv(TABLES_DIR / '01_data_summary.csv', index=False)
by_type.to_csv(TABLES_DIR / '02_event_types.csv', index=False)
by_subtype.to_csv(TABLES_DIR / '02b_protest_sub_event_types.csv', index=False)
by_split.to_csv(TABLES_DIR / '02c_train_test_split_summary.csv', index=False)

monthly = df.groupby('month').size().rename('events').reset_index()
fig, ax = plt.subplots(figsize=(11,4))
ax.plot(monthly['month'], monthly['events'], color='#2F5D8C', linewidth=1.8)
ax.axvline(TEST_START, color='#B04A3A', linestyle='--', linewidth=1.2, label='Test period starts')
ax.set_title('ACLED protest events per month, Africa 2000-2020')
ax.set_xlabel('Month')
ax.set_ylabel('Protest events')
ax.legend()
fig.tight_layout()
save_and_show(fig, 'events_per_month.png', dpi=160)


## 2. Grid: approximate kilometer cells

This first version uses a simple kilometer approximation to avoid heavy GIS setup. It is useful for model development, but a later publication-grade version should use an equal-area projection, for example with GeoPandas.

In [ ]:
def add_grid_columns(events: pd.DataFrame, grid_km: int) -> pd.DataFrame:
    out = events.copy()
    lat0 = out['latitude'].median()
    km_per_degree_lat = 111.0
    km_per_degree_lon = 111.0 * math.cos(math.radians(lat0))
    out['x_km'] = out['longitude'] * km_per_degree_lon
    out['y_km'] = out['latitude'] * km_per_degree_lat
    out['grid_x'] = np.floor(out['x_km'] / grid_km).astype(int)
    out['grid_y'] = np.floor(out['y_km'] / grid_km).astype(int)
    out['cell_id'] = out['grid_x'].astype(str) + '_' + out['grid_y'].astype(str)
    out['grid_km'] = grid_km
    return out

df_grid = add_grid_columns(df.dropna(subset=['latitude','longitude','month']), PRIMARY_GRID_KM)
print('Grid cells with at least one event:', df_grid['cell_id'].nunique())
display(df_grid[['event_date','country','event_type','fatalities','latitude','longitude','grid_x','grid_y','cell_id']].head())

## 3. Build the cell-month panel and features

The panel includes cells that had at least one reported protest during 2000-2020. This first-stage restriction avoids oceans, unpopulated areas, and a very large number of irrelevant zero observations.


In [ ]:
def build_panel(events: pd.DataFrame, grid_km: int, include_sub_event_types=True) -> pd.DataFrame:
    e = add_grid_columns(events.dropna(subset=['latitude','longitude','month']), grid_km)
    e = e[(e['event_date'] >= ANALYSIS_START) & (e['event_date'] <= ANALYSIS_END)].copy()
    e['fatal_protest'] = (pd.to_numeric(e['fatal_protest'], errors='coerce').fillna(0) >= 1).astype(int)
    e['nonfatal_protest'] = (e['fatal_protest'] == 0).astype(int)

    agg = e.groupby(['cell_id','grid_x','grid_y','month']).agg(
        events=('event_id_cnty','count'),
        fatalities=('fatalities','sum'),
        fatal_protests=('fatal_protest','sum'),
        nonfatal_protests=('nonfatal_protest','sum'),
        modal_country=('country', lambda s: s.mode().iat[0] if not s.mode().empty else np.nan)
    ).reset_index()

    if include_sub_event_types:
        subtype_counts = e.pivot_table(index=['cell_id','month'], columns='sub_event_type', values='event_id_cnty', aggfunc='count', fill_value=0).reset_index()
        subtype_counts.columns = ['cell_id','month'] + [f"subtype_{str(c).lower().replace(' ', '_').replace('/', '_')}" for c in subtype_counts.columns[2:]]
        agg = agg.merge(subtype_counts, on=['cell_id','month'], how='left')

    cells = agg[['cell_id','grid_x','grid_y']].drop_duplicates()
    months = pd.date_range(e['month'].min(), e['month'].max(), freq='MS')
    panel = cells.merge(pd.DataFrame({'month': months}), how='cross')
    panel = panel.merge(agg, on=['cell_id','grid_x','grid_y','month'], how='left')

    count_cols = ['events','fatalities','fatal_protests','nonfatal_protests'] + [c for c in panel.columns if c.startswith('subtype_')]
    panel[count_cols] = panel[count_cols].fillna(0)
    panel['modal_country'] = panel.groupby('cell_id')['modal_country'].ffill().bfill()
    panel['protest'] = (panel['events'] > 0).astype(int)
    panel = panel.sort_values(['cell_id','month']).reset_index(drop=True)

    for lag in [1,2,3,6,12]:
        panel[f'events_lag_{lag}'] = panel.groupby('cell_id')['events'].shift(lag).fillna(0)
        panel[f'fatalities_lag_{lag}'] = panel.groupby('cell_id')['fatalities'].shift(lag).fillna(0)
        panel[f'protest_lag_{lag}'] = panel.groupby('cell_id')['protest'].shift(lag).fillna(0)
        panel[f'fatal_protests_lag_{lag}'] = panel.groupby('cell_id')['fatal_protests'].shift(lag).fillna(0)
        panel[f'nonfatal_protests_lag_{lag}'] = panel.groupby('cell_id')['nonfatal_protests'].shift(lag).fillna(0)

    panel['events_roll_6'] = panel.groupby('cell_id')['events'].transform(lambda s: s.shift(1).rolling(6, min_periods=1).sum()).fillna(0)
    panel['events_roll_12'] = panel.groupby('cell_id')['events'].transform(lambda s: s.shift(1).rolling(12, min_periods=1).sum()).fillna(0)
    panel['fatal_protests_roll_6'] = panel.groupby('cell_id')['fatal_protests'].transform(lambda s: s.shift(1).rolling(6, min_periods=1).sum()).fillna(0)
    panel['nonfatal_protests_roll_6'] = panel.groupby('cell_id')['nonfatal_protests'].transform(lambda s: s.shift(1).rolling(6, min_periods=1).sum()).fillna(0)

    base = panel[['grid_x','grid_y','month','events','fatalities','protest','fatal_protests','nonfatal_protests']].copy()
    shifted_neighbors = []
    for dx in [-1,0,1]:
        for dy in [-1,0,1]:
            if dx == 0 and dy == 0:
                continue
            tmp = base.copy()
            tmp['grid_x'] = tmp['grid_x'] - dx
            tmp['grid_y'] = tmp['grid_y'] - dy
            shifted_neighbors.append(tmp)
    neigh = pd.concat(shifted_neighbors, ignore_index=True)
    neigh_agg = neigh.groupby(['grid_x','grid_y','month']).agg(
        neighbor_protests=('events','sum'),
        neighbor_fatalities=('fatalities','sum'),
        neighbor_protest_cells=('protest','sum'),
        neighbor_fatal_protests=('fatal_protests','sum'),
        neighbor_nonfatal_protests=('nonfatal_protests','sum')
    ).reset_index()
    neigh_agg['month'] = neigh_agg['month'] + pd.offsets.MonthBegin(1)
    panel = panel.merge(neigh_agg, on=['grid_x','grid_y','month'], how='left')
    for c in ['neighbor_protests','neighbor_fatalities','neighbor_protest_cells','neighbor_fatal_protests','neighbor_nonfatal_protests']:
        panel[c] = panel[c].fillna(0)

    panel['target_protest_next_month'] = panel.groupby('cell_id')['protest'].shift(-1)
    panel['target_events_next_month'] = panel.groupby('cell_id')['events'].shift(-1)
    panel['month_num'] = panel['month'].dt.month
    panel['year'] = panel['month'].dt.year
    panel['sin_month'] = np.sin(2*np.pi*panel['month_num']/12)
    panel['cos_month'] = np.cos(2*np.pi*panel['month_num']/12)
    return panel.dropna(subset=['target_protest_next_month']).reset_index(drop=True)

panel100 = build_panel(df, PRIMARY_GRID_KM)
print(panel100.shape)
print('Cells:', panel100['cell_id'].nunique(), '| positive target share:', round(panel100['target_protest_next_month'].mean(), 4))
display(panel100.head())
panel100.to_csv(PANELS_DIR / 'panel_100km_month.csv', index=False)


In [ ]:
panel_summary = pd.DataFrame({
    'metric': ['grid_km','cell_months','cells','months','positive_target_share','mean_events_when_positive'],
    'value': [PRIMARY_GRID_KM, len(panel100), panel100['cell_id'].nunique(), panel100['month'].nunique(), round(panel100['target_protest_next_month'].mean(), 4), round(panel100.loc[panel100['events']>0, 'events'].mean(), 2)]
})
display(panel_summary)
panel_summary.to_csv(TABLES_DIR / '03_panel_100km_summary.csv', index=False)

fig, axes = plt.subplots(1,2,figsize=(12,4))
panel100.groupby('month')['protest'].mean().plot(ax=axes[0], color='#2F5D8C')
axes[0].axvline(TEST_START, color='#B04A3A', linestyle='--', linewidth=1.2)
axes[0].set_title('Share of active protest cells per month')
axes[0].set_ylabel('Share with >=1 protest')
panel100.groupby('cell_id')['events'].sum().clip(upper=100).hist(ax=axes[1], bins=30, color='#C8893D')
axes[1].set_title('Distribution of total cell protests, clipped at 100')
axes[1].set_xlabel('Protests per cell, 2000-2020')
fig.tight_layout()
save_and_show(fig, 'panel_diagnostics_100km.png', dpi=160)


## 4. Modeling strategy

We compare five feature sets:

- **A_place_time:** location and time
- **B_local_history:** location/time plus previous protests in the same cell at 1-month, 2-month, 6-month, and 1-year lags
- **C_local_plus_neighbors:** local history plus protests in the eight neighboring cells in the previous month
- **D_fatality_split:** C plus separate neighboring-cell counts for nonfatal and fatal protests
- **E_extended:** D plus local fatal/nonfatal history and protest subtype variables

H1 is tested by whether C improves on B. H3 is tested by comparing the standardized logit coefficients for neighboring nonfatal and fatal protests.


In [ ]:
BASE_FEATURES = ['grid_x','grid_y','year','sin_month','cos_month']
LOCAL_FEATURES = BASE_FEATURES + [
    'events_lag_1','events_lag_2','events_lag_6','events_lag_12',
    'protest_lag_1','protest_lag_2','protest_lag_6','protest_lag_12',
    'events_roll_6','events_roll_12'
]
SPATIAL_FEATURES = LOCAL_FEATURES + ['neighbor_protests','neighbor_protest_cells']
FATALITY_SPLIT_FEATURES = SPATIAL_FEATURES + ['neighbor_nonfatal_protests','neighbor_fatal_protests']
SUBTYPE_FEATURES = [c for c in panel100.columns if c.startswith('subtype_')]
EXTENDED_FEATURES = FATALITY_SPLIT_FEATURES + [
    'neighbor_fatalities',
    'fatalities_lag_1','fatalities_lag_6','fatalities_lag_12',
    'fatal_protests_lag_1','fatal_protests_lag_6','fatal_protests_lag_12','fatal_protests_roll_6',
    'nonfatal_protests_lag_1','nonfatal_protests_lag_6','nonfatal_protests_lag_12','nonfatal_protests_roll_6'
] + SUBTYPE_FEATURES
FEATURE_SETS = {
    'A_place_time': BASE_FEATURES,
    'B_local_history': LOCAL_FEATURES,
    'C_local_plus_neighbors': SPATIAL_FEATURES,
    'D_fatality_split': FATALITY_SPLIT_FEATURES,
    'E_extended': EXTENDED_FEATURES,
}


def make_masks(panel):
    train = panel['month'] <= TRAIN_END
    test = (panel['month'] >= TEST_START) & (panel['month'] <= TEST_END)
    return train, test


train_mask, test_mask = make_masks(panel100)
print('Train rows:', train_mask.sum(), 'Test rows:', test_mask.sum())
print('Train positive share:', round(panel100.loc[train_mask, 'target_protest_next_month'].mean(), 4))
print('Test positive share:', round(panel100.loc[test_mask, 'target_protest_next_month'].mean(), 4))


In [ ]:
def evaluate_predictions(y_true, y_prob, threshold=None):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob)
    if threshold is None:
        threshold = y_true.mean()
    y_pred = (y_prob >= threshold).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary', zero_division=0)
    return {
        'roc_auc': roc_auc_score(y_true, y_prob),
        'avg_precision': average_precision_score(y_true, y_prob),
        'brier': brier_score_loss(y_true, y_prob),
        'threshold': threshold,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'positive_rate_pred': y_pred.mean(),
    }


def fit_models(panel, feature_sets):
    train_mask, test_mask = make_masks(panel)
    y_train = panel.loc[train_mask, 'target_protest_next_month'].astype(int)
    y_test = panel.loc[test_mask, 'target_protest_next_month'].astype(int)
    predictions = panel.loc[test_mask, ['cell_id','grid_x','grid_y','month','target_protest_next_month','events','neighbor_protests','neighbor_nonfatal_protests','neighbor_fatal_protests']].copy()
    results = []

    for name, features in feature_sets.items():
        features = [f for f in features if f in panel.columns]
        X_train = panel.loc[train_mask, features].fillna(0)
        X_test = panel.loc[test_mask, features].fillna(0)

        logit = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE))
        logit.fit(X_train, y_train)
        p_logit = logit.predict_proba(X_test)[:,1]
        row = {'model': f'Logit_{name}', 'feature_set': name, 'n_features': len(features)}
        row.update(evaluate_predictions(y_test, p_logit))
        results.append(row)
        predictions[f'p_Logit_{name}'] = p_logit

        rf = RandomForestClassifier(n_estimators=100, min_samples_leaf=10, class_weight='balanced_subsample', n_jobs=-1, random_state=RANDOM_STATE)
        rf.fit(X_train, y_train)
        p_rf = rf.predict_proba(X_test)[:,1]
        row = {'model': f'RF_{name}', 'feature_set': name, 'n_features': len(features)}
        row.update(evaluate_predictions(y_test, p_rf))
        results.append(row)
        predictions[f'p_RF_{name}'] = p_rf

    return pd.DataFrame(results).sort_values(['avg_precision','roc_auc'], ascending=False), predictions


results100, pred100 = fit_models(panel100, FEATURE_SETS)
display(results100)
results100.to_csv(TABLES_DIR / '04_model_results_100km.csv', index=False)
pred100.to_csv(PANELS_DIR / 'predictions_100km_test.csv', index=False)


## 5. Main interpretation: do neighboring cells add predictive information?


In [ ]:
comparison = results100.copy()
comparison['model_family'] = comparison['model'].str.extract(r'^(Logit|RF)_')
pivot = comparison.pivot_table(index=['model_family','feature_set'], values=['roc_auc','avg_precision','brier','f1'], aggfunc='first').reset_index()
display(pivot)
pivot.to_csv(TABLES_DIR / '05_feature_set_comparison_100km.csv', index=False)

fig, ax = plt.subplots(figsize=(10,4.5))
plot_df = results100.sort_values('avg_precision', ascending=True)
ax.barh(plot_df['model'], plot_df['avg_precision'], color='#2F5D8C')
ax.set_title('Test performance: Average Precision, 100 km grid')
ax.set_xlabel('Average precision')
ax.set_ylabel('')
fig.tight_layout()
save_and_show(fig, 'model_average_precision_100km.png', dpi=160)


## 6. Threshold, ranking, calibration, and H3 diagnostics

F1 is highly sensitive to the classification threshold. In rare-event prediction, the model can have useful ranking ability even when a default threshold gives weak F1. This section adds more appropriate diagnostics:

- **Precision@top 1%, 5%, and 10%** of highest-risk cell-months
- **PR curves** for model ranking under class imbalance
- **Calibration curves** to compare predicted risk with observed protest rates
- **F1-optimal threshold**, separated from the base-rate threshold used above
- **Neighbor uplift table**, comparing B versus C directly
- **H3 coefficient table**, comparing nonfatal and fatal neighboring protest spillovers in a standardized logistic regression


In [ ]:
def threshold_sweep(y_true, y_prob, n_grid=501):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob)
    thresholds = np.linspace(0, 1, n_grid)
    rows = []
    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)
        precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary', zero_division=0)
        rows.append({'threshold': t, 'precision': precision, 'recall': recall, 'f1': f1, 'positive_rate_pred': y_pred.mean()})
    return pd.DataFrame(rows)


def precision_at_top_k(y_true, y_prob, top_shares=(0.01, 0.05, 0.10)):
    d = pd.DataFrame({'y_true': np.asarray(y_true).astype(int), 'y_prob': np.asarray(y_prob)})
    d = d.sort_values('y_prob', ascending=False).reset_index(drop=True)
    rows = []
    n = len(d)
    for share in top_shares:
        k = max(1, int(np.ceil(n * share)))
        top = d.iloc[:k]
        rows.append({
            'top_share': share,
            'n_flagged': k,
            'precision_at_top': top['y_true'].mean(),
            'recall_at_top': top['y_true'].sum() / d['y_true'].sum(),
            'lift_vs_base_rate': top['y_true'].mean() / d['y_true'].mean(),
        })
    return pd.DataFrame(rows)


# Choose interpretable comparison models.
y_test = pred100['target_protest_next_month'].astype(int)
model_cols = {
    'RF_B_local_history': 'p_RF_B_local_history',
    'RF_C_local_plus_neighbors': 'p_RF_C_local_plus_neighbors',
    'RF_D_fatality_split': 'p_RF_D_fatality_split',
    'Logit_C_local_plus_neighbors': 'p_Logit_C_local_plus_neighbors',
    'Logit_D_fatality_split': 'p_Logit_D_fatality_split',
}
model_cols = {k: v for k, v in model_cols.items() if v in pred100.columns}

threshold_rows = []
topk_rows = []
for model_name, col in model_cols.items():
    probs = pred100[col]
    sweep = threshold_sweep(y_test, probs)
    best = sweep.loc[sweep['f1'].idxmax()].copy()
    best['model'] = model_name
    best['base_rate'] = y_test.mean()
    threshold_rows.append(best)

    topk = precision_at_top_k(y_test, probs)
    topk.insert(0, 'model', model_name)
    topk_rows.append(topk)

threshold_eval = pd.DataFrame(threshold_rows)[['model','base_rate','threshold','precision','recall','f1','positive_rate_pred']]
topk_eval = pd.concat(topk_rows, ignore_index=True)

display(threshold_eval)
display(topk_eval)
threshold_eval.to_csv(TABLES_DIR / '08_f1_optimal_thresholds_100km.csv', index=False)
topk_eval.to_csv(TABLES_DIR / '09_precision_at_top_k_100km.csv', index=False)

# Neighbor uplift: C minus B for the same model family.
uplift_rows = []
for family in ['RF', 'Logit']:
    b = results100[(results100['model'] == f'{family}_B_local_history')].iloc[0]
    c = results100[(results100['model'] == f'{family}_C_local_plus_neighbors')].iloc[0]
    uplift_rows.append({
        'model_family': family,
        'roc_auc_B': b['roc_auc'],
        'roc_auc_C': c['roc_auc'],
        'delta_roc_auc_C_minus_B': c['roc_auc'] - b['roc_auc'],
        'avg_precision_B': b['avg_precision'],
        'avg_precision_C': c['avg_precision'],
        'delta_avg_precision_C_minus_B': c['avg_precision'] - b['avg_precision'],
        'brier_B': b['brier'],
        'brier_C': c['brier'],
        'delta_brier_C_minus_B': c['brier'] - b['brier'],
    })
neighbor_uplift = pd.DataFrame(uplift_rows)
display(neighbor_uplift)
neighbor_uplift.to_csv(TABLES_DIR / '10_neighbor_uplift_B_vs_C_100km.csv', index=False)

# H3: compare standardized logistic-regression coefficients for fatal vs nonfatal neighboring protests.
h3_features = [f for f in FATALITY_SPLIT_FEATURES if f in panel100.columns]
train_mask, test_mask = make_masks(panel100)
X_train_h3 = panel100.loc[train_mask, h3_features].fillna(0)
y_train_h3 = panel100.loc[train_mask, 'target_protest_next_month'].astype(int)
h3_logit = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE))
h3_logit.fit(X_train_h3, y_train_h3)
coef = h3_logit.named_steps['logisticregression'].coef_[0]
h3_coef = pd.DataFrame({'feature': h3_features, 'standardized_logit_coefficient': coef})
h3_coef['odds_ratio_per_1sd'] = np.exp(h3_coef['standardized_logit_coefficient'])
h3_focus = h3_coef[h3_coef['feature'].isin(['neighbor_nonfatal_protests','neighbor_fatal_protests'])].copy()
h3_focus['hypothesis'] = 'H3_neighbor_nonfatal_vs_fatal_spillover'
display(h3_focus)
h3_focus.to_csv(TABLES_DIR / '12_h3_neighbor_fatality_spillover_100km.csv', index=False)


In [ ]:
# Fast plotting helpers: keep the analytical content, but avoid plotting tens of thousands of points.
def thin_curve(x, y, max_points=600):
    x = np.asarray(x)
    y = np.asarray(y)
    if len(x) <= max_points:
        return x, y
    idx = np.unique(np.linspace(0, len(x) - 1, max_points).astype(int))
    return x[idx], y[idx]


def fast_quantile_calibration(y_true, y_prob, n_bins=10):
    d = pd.DataFrame({'y_true': np.asarray(y_true).astype(int), 'y_prob': np.asarray(y_prob)})
    d = d.sort_values('y_prob').reset_index(drop=True)
    d['bin'] = pd.qcut(d.index, q=n_bins, labels=False, duplicates='drop')
    cal = d.groupby('bin', as_index=False).agg(
        mean_predicted_risk=('y_prob', 'mean'),
        observed_protest_rate=('y_true', 'mean'),
        n=('y_true', 'size')
    )
    return cal


base_rate = y_test.mean()

# Precision-recall curves.
fig, ax = plt.subplots(figsize=(8, 5))
ax.axhline(base_rate, color='#777777', linestyle='--', linewidth=1.2, label=f'Base rate ({base_rate:.3f})')
for model_name, col in model_cols.items():
    precision, recall, _ = precision_recall_curve(y_test, pred100[col])
    recall_plot, precision_plot = thin_curve(recall, precision, max_points=600)
    ax.plot(recall_plot, precision_plot, linewidth=1.8, label=model_name)
ax.set_title('Precision-recall curves, test period')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.legend(fontsize=8)
fig.tight_layout()
save_and_show(fig, 'precision_recall_curves_100km.png', dpi=140)

# Calibration curves.
calibration_rows = []
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot([0, 1], [0, 1], color='#777777', linestyle='--', linewidth=1.2, label='Perfect calibration')
for model_name, col in model_cols.items():
    cal = fast_quantile_calibration(y_test, pred100[col], n_bins=10)
    cal.insert(0, 'model', model_name)
    calibration_rows.append(cal)
    ax.plot(cal['mean_predicted_risk'], cal['observed_protest_rate'], marker='o', linewidth=1.6, label=model_name)
ax.set_title('Calibration curves, test period')
ax.set_xlabel('Mean predicted risk')
ax.set_ylabel('Observed protest rate')
ax.legend(fontsize=8)
fig.tight_layout()
save_and_show(fig, 'calibration_curves_100km.png', dpi=140)

calibration_table = pd.concat(calibration_rows, ignore_index=True)
calibration_table.to_csv(TABLES_DIR / '11_calibration_bins_100km.csv', index=False)

# Top-k precision plot using plain matplotlib rather than seaborn.
fig, ax = plt.subplots(figsize=(8, 5))
plot_topk = topk_eval.copy()
plot_topk['top_percent'] = (plot_topk['top_share'] * 100).astype(int).astype(str) + '%'
models = list(plot_topk['model'].unique())
top_levels = list(plot_topk['top_percent'].unique())
x = np.arange(len(top_levels))
width = 0.8 / max(1, len(models))
for j, model_name in enumerate(models):
    vals = plot_topk[plot_topk['model'] == model_name].set_index('top_percent').loc[top_levels, 'precision_at_top']
    ax.bar(x + (j - (len(models)-1)/2) * width, vals.values, width=width, label=model_name)
ax.axhline(base_rate, color='#777777', linestyle='--', linewidth=1.2, label='Base rate')
ax.set_title('Precision among highest-risk cell-months')
ax.set_xlabel('Flagged share of test cell-months')
ax.set_ylabel('Precision')
ax.set_xticks(x)
ax.set_xticklabels(top_levels)
ax.legend(fontsize=8)
fig.tight_layout()
save_and_show(fig, 'precision_at_top_k_100km.png', dpi=140)

print('Saved fast evaluation plots and calibration table.')


## 7. Map-like sanity check

This is not a formal map, but it helps assess whether observed and predicted protest risk form plausible geographic clusters.


In [ ]:
best_model_col = 'p_RF_D_fatality_split' if 'p_RF_D_fatality_split' in pred100.columns else 'p_RF_C_local_plus_neighbors'
risk_map = pred100.groupby(['grid_x','grid_y']).agg(
    observed_rate=('target_protest_next_month','mean'),
    predicted_risk=(best_model_col,'mean'),
    test_months=('month','nunique')
).reset_index()

fig, axes = plt.subplots(1,2,figsize=(12,5),sharex=True,sharey=True)
sc1 = axes[0].scatter(risk_map['grid_x'], risk_map['grid_y'], c=risk_map['observed_rate'], s=18, cmap='magma')
axes[0].set_title('Observed protest rate, test')
plt.colorbar(sc1, ax=axes[0], fraction=0.046)
sc2 = axes[1].scatter(risk_map['grid_x'], risk_map['grid_y'], c=risk_map['predicted_risk'], s=18, cmap='magma')
axes[1].set_title(f'Predicted risk: {best_model_col.replace("p_", "")}')
plt.colorbar(sc2, ax=axes[1], fraction=0.046)
for ax in axes:
    ax.set_xlabel('Grid x')
    ax.set_ylabel('Grid y')
fig.tight_layout()
save_and_show(fig, 'observed_vs_predicted_risk_map_100km.png', dpi=180)
risk_map.to_csv(PANELS_DIR / '06_test_risk_map_100km.csv', index=False)


## 8. Robustness check: 50, 100, and 200 km

We run the same comparison between local history and local + neighboring-cell protests for three grid sizes.


In [ ]:
def run_grid_robustness(grid_sizes=(50,100,200)):
    rows = []
    for g in grid_sizes:
        print(f'Running grid {g} km...')
        p = build_panel(df, g)
        fs = {
            'B_local_history': LOCAL_FEATURES,
            'C_local_plus_neighbors': SPATIAL_FEATURES,
            'D_fatality_split': FATALITY_SPLIT_FEATURES,
        }
        res, _ = fit_models(p, fs)
        res['grid_km'] = g
        res['cells'] = p['cell_id'].nunique()
        res['cell_months'] = len(p)
        _, test_mask_g = make_masks(p)
        res['positive_share_test'] = p.loc[test_mask_g, 'target_protest_next_month'].mean()
        rows.append(res)
    return pd.concat(rows, ignore_index=True)


robustness_results = run_grid_robustness(ROBUSTNESS_GRID_KM)
display(robustness_results.sort_values(['grid_km','model']))
robustness_results.to_csv(TABLES_DIR / '07_robustness_50_100_200km.csv', index=False)

fig, ax = plt.subplots(figsize=(9,4.5))
tmp = robustness_results.copy()
tmp['family'] = tmp['model'].str.extract(r'^(Logit|RF)_')
tmp['feature_label'] = tmp['feature_set']
for (fam, label), sub in tmp.groupby(['family','feature_label']):
    sub = sub.sort_values('grid_km')
    ax.plot(sub['grid_km'], sub['avg_precision'], marker='o', label=f'{fam}: {label}')
ax.set_title('Robustness across grid sizes')
ax.set_xlabel('Grid size, km')
ax.set_ylabel('Average precision')
ax.legend(fontsize=8)
fig.tight_layout()
save_and_show(fig, 'robustness_grid_sizes.png', dpi=160)
